In [1]:
from dataclasses import dataclass
from pathlib import Path

import copy
import random
import time

import numpy as np
import torch
import torch.nn as nn
from scipy.io import loadmat
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader, TensorDataset

from brevitas.nn import QuantConv2d, QuantIdentity, QuantLinear, QuantReLU
from brevitas.quant import (
    Int8ActPerTensorFloat,
    Int8Bias,
    Int8WeightPerTensorFloat,
    Int8WeightPerTensorFixedPoint,
    Uint8ActPerTensorFixedPoint,
    Int8BiasPerTensorFixedPointInternalScaling
)
import sys

sys.path.append('..')
from hls4ml.model.profiling import numerical

%matplotlib inline
np.random.seed(0)
torch.manual_seed(0)

In [2]:
DATA_DIR = Path.cwd() / "data"
DATASET_PATH = DATA_DIR / "sat-6-full.mat"
RESULTS_DIR = Path.cwd() / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

INPUT_SHAPE = (4, 28, 28)  # (C, H, W)
NUM_CLASSES = 6
TEST_SIZE = 0.20
VAL_SPLIT = 0.15
RANDOM_SEED = 42

WEIGHT_BITS = 8
ACT_BITS = 8
BIAS_BITS = 8

BATCH_SIZE = 512
EPOCHS = 50
LR = 1e-3
PATIENCE = 10

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

In [57]:
data = loadmat(DATASET_PATH)

images = data["train_x"]  # (H, W, C, N), uint8
labels = data["train_y"]  # (num_classes, N), one-hot

images = images.transpose(3, 2, 0, 1)  # -> (N, C, H, W)
images = images.astype(np.int8)

labels = labels.argmax(axis=0)  # one-hot -> índice de classe

x_train_val, x_test, y_train_val, y_test = train_test_split(
    images, labels, test_size=TEST_SIZE, random_state=RANDOM_SEED, stratify=labels
)

x_train, x_val, y_train, y_val = train_test_split(
    x_train_val, y_train_val, test_size=VAL_SPLIT, random_state=RANDOM_SEED, stratify=y_train_val
)

In [68]:
class T1Brevitas(nn.Module):
    def __init__(self, in_channels=4, num_classes=6, w_bits=8, a_bits=8, b_bits=8):
        super().__init__()

        self.conv1 = QuantConv2d(
            in_channels, 16, kernel_size=3, stride=2, padding=1, bias=True,
            weight_quant=Int8WeightPerTensorFixedPoint, weight_bit_width=w_bits,
            bias_quant=Int8BiasPerTensorFixedPointInternalScaling, # Int8Bias
            bias_bit_width=b_bits,
            return_quant_tensor=False,
        )
        self.bn1 = nn.BatchNorm2d(16)
        self.act1 = QuantReLU(act_quant=Uint8ActPerTensorFixedPoint, bit_width=a_bits, return_quant_tensor=False)

        self.conv2 = QuantConv2d(
            16, 32, kernel_size=1, stride=1, padding=0, bias=True,
            weight_quant=Int8WeightPerTensorFixedPoint, weight_bit_width=w_bits,
            bias_quant=Int8BiasPerTensorFixedPointInternalScaling, # Int8Bias
            bias_bit_width=b_bits,
            return_quant_tensor=False,
        )
        self.bn2 = nn.BatchNorm2d(32)
        self.act2 = QuantReLU(act_quant=Uint8ActPerTensorFixedPoint, bit_width=a_bits, return_quant_tensor=False)

        self.flatten = nn.Flatten()

        self.fc1 = QuantLinear(
            32 * 14 * 14, 64, bias=True,
            weight_quant=Int8WeightPerTensorFixedPoint, weight_bit_width=w_bits,
            bias_quant=Int8BiasPerTensorFixedPointInternalScaling, # Int8Bias
            bias_bit_width=b_bits,
            return_quant_tensor=False,
        )
        self.bn3 = nn.BatchNorm1d(64)
        self.act3 = QuantReLU(act_quant=Uint8ActPerTensorFixedPoint, bit_width=a_bits, return_quant_tensor=False)

        self.fc2 = QuantLinear(
            64, num_classes, bias=True,
            weight_quant=Int8WeightPerTensorFixedPoint, weight_bit_width=w_bits,
            bias_quant=Int8BiasPerTensorFixedPointInternalScaling, # Int8Bias
            bias_bit_width=b_bits,
            return_quant_tensor=False,
        )

    def forward(self, x):
        x = self.act1(self.bn1(self.conv1(x)))
        x = self.act2(self.bn2(self.conv2(x)))
        x = self.flatten(x)
        x = self.act3(self.bn3(self.fc1(x)))
        x = self.fc2(x)
        return x

In [69]:
# sanity check
model = T1Brevitas(w_bits=WEIGHT_BITS, a_bits=ACT_BITS, b_bits=BIAS_BITS)
out = model(torch.randn(2, *INPUT_SHAPE))
print(out.shape)  # esperado: (2, 6)

torch.Size([2, 6])


In [ ]:
# Change the data type of the images to float32 for int8, and adjust the bias type to accept this in training
train_ds = TensorDataset(torch.from_numpy(x_train).float(), torch.from_numpy(y_train).long())
val_ds = TensorDataset(torch.from_numpy(x_val).float(), torch.from_numpy(y_val).long())
test_ds = TensorDataset(torch.from_numpy(x_test).float(), torch.from_numpy(y_test).long())

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

In [77]:
for i,t in train_loader:
    print(i.shape, t.shape)
    print(i.dtype, t.dtype)
    break

torch.Size([512, 4, 28, 28]) torch.Size([512])
torch.float32 torch.int64


In [78]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

model = T1Brevitas(w_bits=WEIGHT_BITS, a_bits=ACT_BITS, b_bits=BIAS_BITS).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

best_val_acc = -1.0
best_state = None
bad_epochs = 0

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()

    model.train()
    train_loss, train_correct, train_total = 0.0, 0, 0
    for images, targets in train_loader:
        images, targets = images.to(device), targets.to(device)

        optimizer.zero_grad()

        logits = model(images)
        logits = logits.value if hasattr(logits, "value") else logits

        loss = criterion(logits, targets)

        loss.backward()
        optimizer.step()

        train_loss += loss.item() * images.size(0)
        train_correct += (logits.argmax(1) == targets).sum().item()
        train_total += images.size(0)
    train_loss /= train_total
    train_acc = train_correct / train_total

    model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    with torch.no_grad():
        for images, targets in val_loader:
            images, targets = images.to(device), targets.to(device)

            logits = model(images)
            logits = logits.value if hasattr(logits, "value") else logits

            loss = criterion(logits, targets)

            val_loss += loss.item() * images.size(0)
            val_correct += (logits.argmax(1) == targets).sum().item()
            val_total += images.size(0)
    val_loss /= val_total
    val_acc = val_correct / val_total

    print(
        f"epoch {epoch:3d}/{EPOCHS} | "
        f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} | "
        f"val_loss={val_loss:.4f} val_acc={val_acc:.4f} | "
        f"{time.time() - t0:.1f}s"
    )

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_state = copy.deepcopy(model.state_dict())
        bad_epochs = 0
    else:
        bad_epochs += 1

    if bad_epochs >= PATIENCE:
        print(f"early stopping na época {epoch} (sem melhora por {PATIENCE} épocas)")
        break

model.load_state_dict(best_state)

device: cpu
epoch   1/50 | train_loss=0.1737 train_acc=0.9485 | val_loss=0.0880 val_acc=0.9714 | 21.1s
epoch   2/50 | train_loss=0.0683 train_acc=0.9767 | val_loss=0.0625 val_acc=0.9783 | 13.3s
epoch   3/50 | train_loss=0.0500 train_acc=0.9826 | val_loss=0.0691 val_acc=0.9762 | 13.2s
epoch   4/50 | train_loss=0.0386 train_acc=0.9867 | val_loss=0.0612 val_acc=0.9781 | 13.2s
epoch   5/50 | train_loss=0.0310 train_acc=0.9894 | val_loss=0.0465 val_acc=0.9835 | 12.4s
epoch   6/50 | train_loss=0.0237 train_acc=0.9920 | val_loss=0.0784 val_acc=0.9733 | 12.3s
epoch   7/50 | train_loss=0.0200 train_acc=0.9935 | val_loss=0.0486 val_acc=0.9830 | 12.3s
epoch   8/50 | train_loss=0.0147 train_acc=0.9954 | val_loss=0.0603 val_acc=0.9792 | 12.3s
epoch   9/50 | train_loss=0.0136 train_acc=0.9957 | val_loss=0.0703 val_acc=0.9782 | 12.2s
epoch  10/50 | train_loss=0.0111 train_acc=0.9965 | val_loss=0.0622 val_acc=0.9800 | 12.3s
epoch  11/50 | train_loss=0.0086 train_acc=0.9974 | val_loss=0.0595 val_acc=0.

<All keys matched successfully>

In [ ]:
from sklearn.metrics import accuracy_score

model.eval()
with torch.no_grad():
    y_brevitas = model(torch.FloatTensor(x_test)).numpy()

print('Accuracy (Brevitas float): {}'.format(accuracy_score(np.argmax(y_test, axis=1), np.argmax(y_brevitas, axis=1))))

best val_acc: 0.9835
test_acc:     0.9831


In [ ]:
import qonnx.util.cleanup
import os
from qonnx.core.modelwrapper import ModelWrapper
from qonnx.transformation.gemm_to_matmul import GemmToMatMul
from brevitas.export import export_qonnx
from class_brevitas import t1_quantizedCNN

#os.makedirs('models', exist_ok=True)
raw_path = 'models/t1_8b_raw.onnx'
final_path = 'models/t1_8b.onnx'


model = t1_quantizedCNN(bit_quantization=8)
model.load_state_dict(torch.load('sat6-cnn-t1w8.pt', weights_only=True))
model.eval()
export_qonnx(model, args=torch.randn(1, 4, 28, 28), export_path=raw_path, opset_version=14)
print(f'Exported raw QONNX to {raw_path}')

# Clean and convert Gemm → MatMul
qonnx.util.cleanup.cleanup(raw_path, out_file=final_path)
model_wrapper = ModelWrapper(final_path)
model_wrapper = model_wrapper.transform(GemmToMatMul())
model_wrapper = qonnx.util.cleanup.cleanup_model(model_wrapper)
model_wrapper.save(final_path)
print(f'Cleaned QONNX saved to {final_path}')

TypeError: t1_quantizedCNN.__init__() missing 1 required positional argument: 'bit_quantization'

In [ ]:
import hls4ml
from sklearn.metrics import accuracy_score

config = hls4ml.utils.config_from_onnx_model(
    model_wrapper,
    granularity='name',
    backend='Vitis',
)

hls_model = hls4ml.converters.convert_from_onnx_model(
    model_wrapper,
    output_dir='/hls4ml_prjs/hls4ml_prj_brevitas_part2b',
    backend='Vitis',
    hls_config=config,
    part='xc7z020clg400-1',
)
hls_model.compile()
y_hls = hls_model.predict(np.ascontiguousarray(x_test, dtype=np.float32))

In [ ]:
print('Accuracy Brevitas (float):    {}'.format(accuracy_score(np.argmax(y_test, axis=1), np.argmax(y_brevitas, axis=1))))
print('Accuracy hls4ml:              {}'.format(accuracy_score(np.argmax(y_test, axis=1), np.argmax(y_hls, axis=1))))


In [ ]:
hls_model.build(csim=False)